# Experiment A — Manual Prompt Baseline

**Evolutionary Prompt Optimization for LLM-Based Sentiment Analysis**  
**Member 4 — Experimental Design and Baseline Implementation**

---

## Purpose

This notebook implements **Experiment A** of the three-way comparison:

| Experiment | Method | Description |
|---|---|---|
| **A (this notebook)** | Manual prompting | Evaluate the 8 hand-crafted prompts authored by Member 2 |
| B (`08_…`) | Random generation | Evaluate a random sample of prompts with no evolutionary pressure |
| C (`09_…`) | Genetic Algorithm | Load GA results from Member 3 and compare all three approaches |

### Protocol (identical for every experiment)
- **Dataset**: SST-2 (nyu-mll/glue)
- **Model**: `Qwen/Qwen2.5-1.5B-Instruct` — same revision, same greedy decoding
- **Optimization split** (`opt.csv`): used for prompt scoring / selection only
- **Validation split** (`val.csv`): used **once**, at the end, to report generalisation
- **Test set**: never touched during any experiment
- **Metric**: Macro F1 (primary), Accuracy (secondary)
- **Evaluation budget**: equal number of scored examples across all experiments
- **Seed**: 42 everywhere

## 1. Environment Setup

In [2]:
import os
import sys
import json
import subprocess
import random
import time
import importlib.util
from pathlib import Path

REPO_URL = "https://github.com/sarrazer24/evolutionary-prompt-optimization.git"
REPO_DIR = Path("/kaggle/working/evolutionary-prompt-optimization")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(REPO_DIR)

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "sklearn": "scikit-learn",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
}

missing_packages = [
    pkg for mod, pkg in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(mod) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )
else:
    print("All required packages are already installed.")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Project directory:", Path.cwd())
print("Python version:", sys.version)
print("Setup complete.")

fatal: could not create leading directories of '/kaggle/working/evolutionary-prompt-optimization': Read-only file system


CalledProcessError: Command '['git', 'clone', '--branch', 'main', '--single-branch', 'https://github.com/sarrazer24/evolutionary-prompt-optimization.git', '/kaggle/working/evolutionary-prompt-optimization']' returned non-zero exit status 128.

## 2. Imports and Reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datetime import datetime
from IPython.display import display

from src.data.sst2_data import load_saved_splits, stratified_sample
from src.llm.config import ModelConfig
from src.llm.model_interface import LLMInterface
from src.prompts.baseline_prompts import get_baseline_prompts
from src.evaluation.evaluation import PredictionCache, evaluate_prompt, save_results

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Global seed: {SEED}")

## 3. GPU Check

In [ ]:
if not torch.cuda.is_available():
    print("⚠  No GPU detected — inference will be very slow on CPU.")
    print("   On Kaggle: Settings → Accelerator → GPU T4 x2.")
else:
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i}: {name}  ({mem:.1f} GB)")

## 4. Experiment Configuration

All hyperparameters are defined here as named constants so the notebook is fully reproducible and easy to audit.

In [ ]:
# ── Experiment A configuration ────────────────────────────────────────────────

# Evaluation budget: number of optimisation-split examples scored per prompt.
# Keep identical to Experiments B and C for a fair comparison.
N_OPT_EXAMPLES   = 1_000   # examples used for per-prompt scoring on opt split
N_VAL_EXAMPLES   = 1_000   # examples used for final validation (held-out)

# Output directory (relative to repo root, matches experiments/results/ convention)
OUTPUT_DIR = Path("experiments/results/experiment_A_manual")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_CONFIG = {
    "experiment": "A",
    "description": "Manual prompt baseline — 8 hand-crafted prompts by Member 2",
    "dataset": "nyu-mll/glue:sst2",
    "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
    "inference": {"greedy": True, "max_new_tokens": 8, "batch_size": 32},
    "split_seed": 42,
    "evaluation_seed": SEED,
    "n_opt_examples": N_OPT_EXAMPLES,
    "n_val_examples": N_VAL_EXAMPLES,
    "metric_primary": "macro_f1",
    "metric_secondary": "accuracy",
    "test_set_used": False,
    "created_at": datetime.now().isoformat(),
}

config_path = OUTPUT_DIR / "experiment_config.json"
config_path.write_text(json.dumps(EXPERIMENT_CONFIG, indent=4, ensure_ascii=False))
print(json.dumps(EXPERIMENT_CONFIG, indent=2))

## 5. Load Data

In [ ]:
opt_df, val_df = load_saved_splits()

print(f"Optimisation split : {len(opt_df):>6,} rows  "
      f"(positive: {opt_df['label'].mean():.1%})")
print(f"Validation split   : {len(val_df):>6,} rows  "
      f"(positive: {val_df['label'].mean():.1%})")

# Stratified subsample for scoring (keeps class balance)
rng = random.Random(SEED)
opt_sample = stratified_sample(opt_df, n=N_OPT_EXAMPLES, rng=rng)
val_sample  = stratified_sample(val_df,  n=N_VAL_EXAMPLES,  rng=rng)

opt_texts  = opt_sample["text"].tolist()
opt_labels = opt_sample["label"].tolist()
val_texts  = val_sample["text"].tolist()
val_labels = val_sample["label"].tolist()

print(f"\nOpt sample : {len(opt_texts)} examples  "
      f"(positive: {sum(opt_labels)/len(opt_labels):.1%})")
print(f"Val sample : {len(val_texts)} examples  "
      f"(positive: {sum(val_labels)/len(val_labels):.1%})")

## 6. Load the Manual Prompts (Member 2)

All 8 prompts are defined in `src/prompts/baseline_prompts.py`.

In [ ]:
MANUAL_PROMPTS = get_baseline_prompts()   # list of (name, prompt_text)

print(f"Number of manual prompts: {len(MANUAL_PROMPTS)}")
print()
for i, (name, text) in enumerate(MANUAL_PROMPTS, 1):
    print(f"[{i}] {name}")
    print(f"    {text[:120]}{'…' if len(text) > 120 else ''}")
    print()

## 7. Load the LLM

In [ ]:
model_config = ModelConfig()   # Qwen2.5-1.5B-Instruct, pinned revision, greedy classification
llm = LLMInterface(model_config)
llm.load()
print(f"Model loaded: {llm.model_id}")
print(f"Device      : {llm.device}")

## 8. Evaluate Every Manual Prompt on the Optimisation Split

All prompts are scored on **the same 1 000-example sample** (never the validation or test split).

In [ ]:
cache_A = PredictionCache()
opt_results = []   # one dict per prompt

for idx, (name, prompt) in enumerate(MANUAL_PROMPTS, 1):
    print(f"[{idx}/{len(MANUAL_PROMPTS)}] Evaluating: {name}")
    t0 = time.time()
    result = evaluate_prompt(
        prompt=prompt,
        examples=opt_texts,
        labels=opt_labels,
        predict_fn=llm.predict_raw_batch,
        model_id=llm.model_id,
        inference_config=llm.inference_config,
        cache=cache_A,
        split_name="opt",
    )
    elapsed = time.time() - t0
    m = result["metrics"]
    opt_results.append({
        "rank": None,                     # filled after sorting
        "prompt_name": name,
        "prompt_text": prompt,
        "macro_f1": round(m["f1_macro"], 6),
        "accuracy":  round(m["accuracy"],  6),
        "precision_macro": round(m["precision_macro"], 6),
        "recall_macro":    round(m["recall_macro"],    6),
        "n_invalid":  m["n_invalid"],
        "invalid_rate": round(m["invalid_rate"], 6),
        "n_examples": m["n_examples"],
        "n_cached":   result["n_cached"],
        "split": "opt",
        "elapsed_s": round(elapsed, 2),
    })
    print(f"   Macro F1 = {m['f1_macro']:.4f}  |  "
          f"Accuracy = {m['accuracy']:.4f}  |  "
          f"Invalid = {m['n_invalid']}  |  {elapsed:.1f}s")

# Sort by Macro F1 descending and assign ranks
opt_results.sort(key=lambda r: r["macro_f1"], reverse=True)
for rank, r in enumerate(opt_results, 1):
    r["rank"] = rank

print("\nAll manual prompts evaluated.")

## 9. Optimisation-Split Leaderboard

In [ ]:
df_opt = pd.DataFrame(opt_results)[
    ["rank", "prompt_name", "macro_f1", "accuracy", "n_invalid", "invalid_rate", "elapsed_s"]
]
display(df_opt.set_index("rank"))

best_name   = opt_results[0]["prompt_name"]
best_prompt = opt_results[0]["prompt_text"]
best_f1_opt = opt_results[0]["macro_f1"]
print(f"\n✔ Best manual prompt on opt split: '{best_name}'  (Macro F1 = {best_f1_opt:.4f})")

## 10. Visualise Optimisation Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names  = [r["prompt_name"] for r in opt_results]
f1s    = [r["macro_f1"]    for r in opt_results]
accs   = [r["accuracy"]    for r in opt_results]
colors = ["#2ecc71" if i == 0 else "#3498db" for i in range(len(names))]

# Macro F1 bar chart
axes[0].barh(names[::-1], f1s[::-1], color=colors[::-1])
axes[0].set_xlabel("Macro F1")
axes[0].set_title("Experiment A — Manual Prompts\n(opt split, ranked by Macro F1)")
axes[0].set_xlim(max(0, min(f1s) - 0.02), min(1, max(f1s) + 0.02))
for v, y in zip(f1s[::-1], range(len(names))):
    axes[0].text(v + 0.001, y, f"{v:.4f}", va="center", fontsize=8)

# Accuracy vs Macro F1 scatter
axes[1].scatter(f1s, accs, s=80, zorder=3)
for name, x, y in zip(names, f1s, accs):
    axes[1].annotate(name, (x, y), textcoords="offset points",
                     xytext=(5, 3), fontsize=7)
axes[1].set_xlabel("Macro F1")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy vs Macro F1 (opt split)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = OUTPUT_DIR / "manual_prompts_opt.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {plot_path}")

## 11. Validate the Best Manual Prompt

Validation is run **exactly once** on a held-out sample from `val.csv`.  
This score is later used in Notebook 09 for the three-way comparison.

In [ ]:
print(f"Validating best prompt: '{best_name}'")
print(f"Prompt text: {best_prompt}\n")

t0 = time.time()
val_result = evaluate_prompt(
    prompt=best_prompt,
    examples=val_texts,
    labels=val_labels,
    predict_fn=llm.predict_raw_batch,
    model_id=llm.model_id,
    inference_config=llm.inference_config,
    split_name="val",
)
elapsed_val = time.time() - t0

vm = val_result["metrics"]
val_summary = {
    "prompt_name":     best_name,
    "prompt_text":     best_prompt,
    "macro_f1":        round(vm["f1_macro"],       6),
    "accuracy":        round(vm["accuracy"],        6),
    "precision_macro": round(vm["precision_macro"], 6),
    "recall_macro":    round(vm["recall_macro"],    6),
    "n_invalid":       vm["n_invalid"],
    "invalid_rate":    round(vm["invalid_rate"],    6),
    "n_examples":      vm["n_examples"],
    "split":           "val",
    "elapsed_s":       round(elapsed_val, 2),
}

print(f"Validation Macro F1 : {vm['f1_macro']:.4f}")
print(f"Validation Accuracy : {vm['accuracy']:.4f}")
print(f"Invalid rate        : {vm['invalid_rate']:.2%}")
print(f"Elapsed             : {elapsed_val:.1f}s")

## 12. Save Results

In [ ]:
# ── Full opt-split results (CSV + JSON) ───────────────────────────────────────
opt_csv_path  = OUTPUT_DIR / "experiment_A_opt_results.csv"
opt_json_path = OUTPUT_DIR / "experiment_A_opt_results.json"

save_results(opt_results, opt_csv_path,  append=False)
save_results(opt_results, opt_json_path)
print(f"Opt results  → {opt_csv_path}")
print(f"             → {opt_json_path}")

# ── Validation summary ────────────────────────────────────────────────────────
val_csv_path  = OUTPUT_DIR / "experiment_A_val_best.csv"
val_json_path = OUTPUT_DIR / "experiment_A_val_best.json"

save_results(val_summary, val_csv_path,  append=False)
save_results(val_summary, val_json_path)
print(f"Val results  → {val_csv_path}")
print(f"             → {val_json_path}")

# ── Comparison-ready summary (consumed by Notebook 09) ───────────────────────
comparison_payload = {
    "experiment": "A",
    "method": "manual_prompts",
    "n_candidates": len(MANUAL_PROMPTS),
    "n_opt_examples": N_OPT_EXAMPLES,
    "n_val_examples": N_VAL_EXAMPLES,
    "best_prompt_name": best_name,
    "best_prompt_text": best_prompt,
    "best_macro_f1_opt": best_f1_opt,
    "best_accuracy_opt": opt_results[0]["accuracy"],
    "best_macro_f1_val": val_summary["macro_f1"],
    "best_accuracy_val": val_summary["accuracy"],
    "all_opt_results": opt_results,
    "model_id": llm.model_id,
    "model_revision": model_config.revision,
    "seed": SEED,
    "created_at": datetime.now().isoformat(),
}

summary_path = OUTPUT_DIR / "experiment_A_summary.json"
summary_path.write_text(json.dumps(comparison_payload, indent=4, ensure_ascii=False, default=str))
print(f"Summary      → {summary_path}")

print("\nAll files saved.")

## 13. Final Summary

In [ ]:
print("=" * 60)
print("EXPERIMENT A — MANUAL PROMPT BASELINE")
print("=" * 60)
print(f"Prompts evaluated       : {len(MANUAL_PROMPTS)}")
print(f"Opt examples per prompt : {N_OPT_EXAMPLES}")
print(f"Val examples (best)     : {N_VAL_EXAMPLES}")
print()
print(f"Best prompt             : {best_name}")
print(f"  Opt  Macro F1         : {best_f1_opt:.4f}")
print(f"  Val  Macro F1         : {val_summary['macro_f1']:.4f}")
print(f"  Val  Accuracy         : {val_summary['accuracy']:.4f}")
print()
print("Output files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")
print()
print("Next step: run Notebook 08 (Experiment B — Random Baseline)")
print("Then     : run Notebook 09 (Experiment C — Full Three-Way Comparison)")